# Baselines on the validation terms (2016–2020)

Every model has to beat these two. Each run is appended to `results/micah.csv`.

In [ ]:
import sys
from pathlib import Path

# make the repo root importable so `src` works from notebooks/micah/
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.baselines import always_petitioner, previous_terms_majority
from src.config import FEATURE_COLUMNS
from src.data import add_circuit_features, load_data
from src.evaluate import RESULTS_DIR, evaluate
from src.features import add_features
from src.split import split_by_term

In [ ]:
df = load_data(REPO_ROOT / "data" / "SCDB_2026_01_caseCentered_Citation.csv")
df = add_circuit_features(add_features(df))
X_train, y_train, X_val, y_val, X_test, y_test = split_by_term(df, FEATURE_COLUMNS)

In [ ]:
# Rolling baseline: each validation term is predicted from the 10 terms before it.
# Pass the full filtered df so 2017 can look back at 2016, etc.
majority_pred = previous_terms_majority(df, X_val["term"], window=10)
print("prediction per term:", majority_pred.groupby(X_val["term"]).first().to_dict())

evaluate(y_val, always_petitioner(X_val), model="always petitioner", name="micah", split="val")
evaluate(y_val, majority_pred, model="previous 10 terms majority", name="micah", split="val")

pd.read_csv(RESULTS_DIR / "micah.csv")